# 🤖 Partie 2 : RAG (Retrieval-Augmented Generation) étape par étape

## ❓ Est-ce que ce processus s'appelle du Fine-Tuning ?

**NON ! Ce n'est PAS du Fine-Tuning.** C'est du **RAG**. Voici la différence expliquée très simplement :

### 1. Le Fine-Tuning (L'examen de mémoire) 🧠
- Vous **ré-entraînez** le modèle (vous modifiez ses "neurones"/poids mathématiques).
- C'est comme obliger un étudiant à apprendre par cœur 5000 pages d'un livre pour un examen.
- **Problèmes** : Ça coûte très cher (besoin de grosses cartes graphiques GPU), ça prend du temps, et le modèle peut quand même oublier ou halluciner (inventer des choses).

### 2. Le RAG (L'examen à livre ouvert) 📖
- Le modèle **ne change pas**, on ne l'entraîne pas du tout.
- Quand vous posez une question, le système de recherche (FAISS) va chercher les 3 paragraphes les plus pertinents dans vos documents, et les donne au modèle en disant : *"Lis ce texte et réponds à la question en utilisant uniquement ces informations"*.
- **Avantages** : C'est gratuit (pas d'entraînement), immédiat, et le modèle ne dit pas de bêtises car il copie la réponse de vos documents.

---

## Étape 1 : Installation et Importations
Nous avons besoin des bibliothèques pour lire les fichiers, créer les vecteurs, et générer la réponse.

In [ ]:
!pip install sentence-transformers faiss-cpu pypdf numpy tqdm transformers torch

In [ ]:
import os
import glob
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import time

## Étape 2 : Chargement et découpage (Chunking) des documents
On ne peut pas donner un livre entier à lire au modèle d'un coup. On doit donc découper nos documents en petits morceaux de ~500 caractères, qu'on appelle des **chunks**.

In [ ]:
def decouper_texte(texte, chunk_size=500, chunk_overlap=50):
    chunks = []
    debut = 0
    while debut < len(texte):
        fin = debut + chunk_size
        morceau = texte[debut:fin]
        
        # Essayer de couper à la fin d'une phrase (au point)
        if fin < len(texte):
            dernier_point = morceau.rfind('.')
            if dernier_point > chunk_size // 2:
                fin = debut + dernier_point + 1
                morceau = texte[debut:fin]
        
        if morceau.strip():
            chunks.append(morceau.strip())
        
        debut = fin - chunk_overlap
    return chunks

def charger_documents(dossier):
    chunks_finaux = []
    fichiers = glob.glob(os.path.join(dossier, "*.txt"))
    
    print(f"Trouvé {len(fichiers)} documents dans {dossier}")
    
    for fichier in fichiers:
        nom_fichier = os.path.basename(fichier)
        with open(fichier, 'r', encoding='utf-8') as f:
            texte = f.read()
        
        morceaux = decouper_texte(texte)
        for i, morceau in enumerate(morceaux):
            chunks_finaux.append({
                "source": nom_fichier,
                "texte": morceau
            })
        print(f"- {nom_fichier} découpé en {len(morceaux)} morceaux.")
        
    return chunks_finaux

# Exécution du chargement
dossier_docs = "documents"
mes_chunks = charger_documents(dossier_docs)
print(f"\nTotal : {len(mes_chunks)} petits morceaux de texte (chunks) créés.")

## Étape 3 : Création des Vecteurs (Embeddings)
Nous allons transformer chaque texte en une liste de nombres (vecteur) grâce à `sentence-transformers`.

In [ ]:
print("Chargement du modèle d'embeddings...")
encodeur = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

textes_bruts = [chunk["texte"] for chunk in mes_chunks]

print(f"Transformation des {len(textes_bruts)} textes en nombres (embeddings)...")
embeddings = encodeur.encode(textes_bruts, convert_to_numpy=True, show_progress_bar=True)

print(f"Taille de la matrice : {embeddings.shape} (Chaque texte est devenu une suite de nombres)")

## Étape 4 : Base de données FAISS
On met tous ces vecteurs dans FAISS, le moteur de recherche hyper-rapide créé par Meta.

In [ ]:
dimension = embeddings.shape[1]
index_faiss = faiss.IndexFlatL2(dimension)
index_faiss.add(embeddings.astype('float32'))

print(f"{index_faiss.ntotal} documents stockés dans le radar FAISS.")

## Étape 5 : Le Retriever (Moteur de recherche)
Maintenant, quand on pose une question, on demande à FAISS de trouver les textes les plus proches mathématiquement de la question.

In [ ]:
def rechercher_documents(question, top_k=3):
    vecteur_question = encodeur.encode([question], convert_to_numpy=True).astype('float32')
    distances, indices = index_faiss.search(vecteur_question, top_k)
    
    resultats = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx < len(mes_chunks):
            resultats.append(mes_chunks[idx])
    return resultats

question_test = "Qu'est-ce que le mécanisme d'attention ?"
docs_trouves = rechercher_documents(question_test)

print(f"Question : {question_test}\n")
for i, doc in enumerate(docs_trouves):
    print(f"--- Résultat {i+1} (Source: {doc['source']}) ---")
    print(doc['texte'][:150] + "...\n")

## Étape 6 : Le Prompt Augmenté
On combine la question ET les documents trouvés pour créer la consigne pour l'IA.

In [ ]:
def construire_prompt(question, documents_trouves):
    contexte = ""
    for doc in documents_trouves:
        contexte += doc['texte'] + "\n\n"
        
    prompt = (
        "Utilise uniquement le contexte suivant pour répondre à la question. "
        "Si la réponse n'y est pas, dis que tu ne sais pas.\n\n"
        f"CONTEXTE :\n{contexte}\n"
        f"QUESTION : {question}\n\n"
        "RÉPONSE :"
    )
    return prompt

prompt_final = construire_prompt(question_test, docs_trouves)
print(prompt_final[:300] + "... [Prompt tronqué]")

## Étape 7 : Le Générateur (LLM)
On donne ce "prompt augmenté" à un LLM gratuit (Flan-T5) pour générer la réponse finale. (C'est l'étudiant qui répond avec le livre ouvert).

In [ ]:
print("Chargement du LLM Flan-T5...")
tokenizer_llm = AutoTokenizer.from_pretrained("google/flan-t5-base")
modele_llm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def generer_reponse(prompt):
    inputs = tokenizer_llm(prompt, return_tensors="pt", max_length=512, truncation=True)
    with torch.no_grad():
        outputs = modele_llm.generate(
            **inputs, 
            max_new_tokens=150, 
            temperature=0.3,
            do_sample=True
        )
    return tokenizer_llm.decode(outputs[0], skip_special_tokens=True)

print("\n" + "="*50)
print(f"QUESTION : {question_test}")
print("="*50)
reponse = generer_reponse(prompt_final)
print(f"\nRÉPONSE :\n{reponse}")